# WAPL stage 73 — offline alignment filtering of Arabic web text toward ArabicMMLU (Qwen2.5-1.5B)

Protocol: `research/llm-web-filter-protocol.json` (amended before any outcome; see `research/llm-web-filter-deviations.md`).

Runtime → **L4 GPU**. The run is resumable from the journal on Google Drive. About 3 hours on an L4.

شغّلي الخلايا بالترتيب. التشغيل قابل للاستئناف.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
JOURNAL = '/content/drive/MyDrive/wapl/stage73-journal.jsonl'

In [ ]:
%cd /content
!rm -rf WAPL-Research && git clone -q https://github.com/4hessa/WAPL-Research.git
%cd /content/WAPL-Research
!git log --oneline -1

In [ ]:
!pip install -q transformers==4.46.3 tokenizers==0.20.3 huggingface_hub==0.26.2
import torch, transformers; print(torch.__version__, torch.version.cuda, transformers.__version__, torch.cuda.get_device_name(0))

In [ ]:
%cd /content/WAPL-Research
import json
from huggingface_hub import snapshot_download, hf_hub_download
spec = json.load(open('research/llm-gpu-scale-protocol.json'))['models']['qwen2.5-1.5b']
snapshot_download(spec['repo'], revision=spec['revision'], local_dir='models/qwen2.5-1.5b',
                  allow_patterns=['*.json', '*.safetensors', '*.txt'])
FW = 'af9c13333eb981300149d5ca60a8e9d659b276b9'
KEPT = hf_hub_download('HuggingFaceFW/fineweb-2', 'data/arb_Arab/test/000_00000.parquet', repo_type='dataset', revision=FW, local_dir='/content/stage73_data/web')
REMOVED = hf_hub_download('HuggingFaceFW/fineweb-2', 'data/arb_Arab_removed/train/000_00000.parquet', repo_type='dataset', revision=FW, local_dir='/content/stage73_data/web')  # 4.8 GB
for f in ('All/dev.csv', 'All/test.csv'):
    hf_hub_download('MBZUAI/ArabicMMLU', f, repo_type='dataset', revision='7aa530e2893ac420352b3f5c1a1310c010e9758b', local_dir='/content/stage73_data/mmlu_repo')
MMLU = '/content/stage73_data/mmlu_repo/All'
print(KEPT, REMOVED, MMLU)

## Run
Checks every hash, calibrates costs once before any evaluation, then runs 24 trajectories. Outcomes are not printed.

In [ ]:
%cd /content/WAPL-Research
!python -m experiments.llm_web_filter --journal {JOURNAL} --kept {KEPT} --removed {REMOVED} --mmlu {MMLU} --summary results/llm-web-filter.json

## Audit (same GPU)

In [ ]:
%cd /content/WAPL-Research
import shutil; shutil.copy(JOURNAL, 'results/llm-web-filter-journal.jsonl')
!python -m tools.audit_llm_web_filter --kept {KEPT} --removed {REMOVED} --mmlu {MMLU}